In [4]:
import pandas as pd
import numpy as np
import os

from PIL import Image
from matplotlib.image import imread
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, silhouette_score
from sklearn.utils.class_weight import compute_class_weight


from tensorflow import keras
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.losses import Huber

import cv2



## Importación de X e y para Computer Vision

In [7]:
X_bn_cv = np.load(os.path.join("datos_procesados/X_bn_cv.npy"))

y_bn_cv = np.load(os.path.join("datos_procesados/y_bn_cv.npy"))

In [8]:
X_bn_cv.shape

(15000, 64, 64, 1)

In [9]:
y_bn_cv.shape

(15000,)

In [10]:
X_cv_train, X_cv_val, y_cv_train, y_cv_val = train_test_split(
    X_bn_cv, 
    y_bn_cv, 
    test_size=0.1, 
    random_state=11
)

print("Tamaño X_train:", X_cv_train.shape)
print("Tamaño X_val:", X_cv_val.shape)
print("Tamaño X_test:", y_cv_train.shape)
print("Tamaño y_val:", y_cv_val.shape)

Tamaño X_train: (13500, 64, 64, 1)
Tamaño X_val: (1500, 64, 64, 1)
Tamaño X_test: (13500,)
Tamaño y_val: (1500,)


## Entrenamiento con Computer Vision

In [ ]:
preds_cv_val = []
for img in X_cv_val:
    # Elimina dimensiones innecesarias: si la imagen es (64, 64, 1), squeeze() la deja en (64, 64)
    # OpenCV necesita una matriz de 2D para aplicar umbrales y contornos.
    img_squeeze = img.squeeze()
    # Aplica desenfoque gaussiano (Gaussian Blur)
    # Suaviza la imagen para eliminar ruido o píxeles sueltos que puedan parecer clips.
    blur = cv2.GaussianBlur(img_squeeze, (5,5), 0) # (5,5) tamaño del filtro; 0 desviación estándar.
    # Binarización (Thresholding)
    # Convierte la imagen a blanco y negro puro (sin grises).
    # Todo píxel mayor a 127 se vuelve 255 (blanco), lo demás 0 (negro).
    _, thresh = cv2.threshold(blur, 127, 255, cv2.THRESH_BINARY)
    # Encuentra Contornos
    # Cada mancha detectada se considera un "clip".
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # Guardar la cantidad de objetos encontrados
    preds_cv_val.append(len(contours))

# Convierte predicciones y valores reales a arrays de NumPy
preds_val_np = np.array(preds_cv_val, dtype=np.int32)
y_val_np = np.array(y_cv_val).reshape(-1)

# Calculo del RMSE (Raíz del Error Cuadrático Medio)
rmse_cv = np.sqrt(np.mean((y_val_np - preds_val_np)**2))
print("RMSE CV:", rmse_cv)


RMSE CV: 42.72558952197149


## Importación y normalización de X e y para CNN

In [11]:
X_rgb_cnn = np.load(os.path.join("datos_procesados/X_rgb_cnn.npy"))

y_rgb_cnn = np.load(os.path.join("datos_procesados/y_rgb_cnn.npy"))

In [12]:
X_rgb_cnn.shape

(15000, 64, 64, 3)

In [13]:
y_rgb_cnn.shape

(15000,)

In [14]:
np.max(X_rgb_cnn)

np.float32(255.0)

In [15]:
np.min(X_rgb_cnn)

np.float32(133.0)

In [16]:
X_st = X_rgb_cnn / 255.0
print(np.max(X_st), np.min(X_st))

1.0 0.52156866


In [17]:
X_cnn_train, X_cnn_val, y_cnn_train, y_cnn_val = train_test_split(
    X_st, y_rgb_cnn,
    test_size=0.1,          
    random_state=11
)

X_cnn_train, y_cnn_train = shuffle(X_cnn_train, y_cnn_train, random_state=11)

print("Tamaño X_train:", X_cnn_train.shape)
print("Tamaño X_val:", X_cnn_val.shape)
print("Tamaño X_test:", y_cnn_train.shape)
print("Tamaño y_val:", y_cnn_val.shape)

Tamaño X_train: (13500, 64, 64, 3)
Tamaño X_val: (1500, 64, 64, 3)
Tamaño X_test: (13500,)
Tamaño y_val: (1500,)


## Modelo CNN

In [ ]:
# Data Augmentation
datagen = ImageDataGenerator(
    zoom_range=0.05,
    width_shift_range=0.05,
    height_shift_range=0.05
)
datagen.fit(X_cnn_train)

In [ ]:
cnn_model = Sequential([

    # Capas convolucionales

    # Padding: marco de píxeles extra para que la imagen no se haga más pequeña tras la convolución y no se pierda información
    # La imagen entra y sale de la capa de convolución el mismo tamaño

    # Capa 1: reconocimiento de formas simples
    Conv2D(32,(3,3),activation='relu', padding='same', input_shape=(64,64,3)),
    BatchNormalization(),

    # Capa 2: Combina las formas simples 
    Conv2D(64,(3,3),activation='relu', padding='same'),
    # Normaliza las activaciones para acelerar el entrenamiento y dar estabilidad
    BatchNormalization(),
    MaxPooling2D(2,2),

    # Capa 3: Extrae patrones complejos
    Conv2D(128,(3,3),activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    # Capas neuronales
    Flatten(),
    Dense(256, activation='relu'),
    # Apaga aleatoriamente el 30% de las neuronas para evitar el sobreajuste (overfitting)
    Dropout(0.3),
    Dense(1)
])

cnn_model.compile(
    optimizer=Adam(learning_rate=0.001), # velocidad a la que el optimizador ajusta los pesos del modelo
    # evita que los outliers desestabilicen el aprendizaje. Usado en problemas de regresión
    loss=Huber()  
)

In [ ]:
# detiene el entrenamiento si el error no mejora en 10 épocas y recupera automáticamente los mejores pesos alcanzados
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [81]:
cnn_model.fit(
    datagen.flow(X_cnn_train, y_cnn_train, batch_size=32),
    validation_data=(X_cnn_val, y_cnn_val),
    epochs=50,
    callbacks=[early_stop]
)

Epoch 1/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 235s 550ms/step - loss: 7.2051 - val_loss: 15.7754
Epoch 2/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 228s 541ms/step - loss: 5.4885 - val_loss: 14.6218
Epoch 3/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 226s 535ms/step - loss: 5.2322 - val_loss: 11.3663
Epoch 4/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 225s 534ms/step - loss: 4.8985 - val_loss: 35.0273
Epoch 5/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 225s 534ms/step - loss: 4.8738 - val_loss: 21.1568
Epoch 6/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 226s 535ms/step - loss: 4.8213 - val_loss: 12.8691
Epoch 7/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 225s 534ms/step - loss: 4.6386 - val_loss: 5.5217
Epoch 8/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 225s 534ms/step - loss: 4.5373 - val_loss: 53.1431
Epoch 9/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 226s 535ms/step - loss: 4.4690 - val_loss: 87.9874
Epoch 10/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 226s 536ms/step - loss: 4.4021 - val_loss: 87.7298
Epoch 11/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 226s 535ms/step - loss: 4.7245 - val_loss: 3.5794

#### Exportación del modelo

In [ ]:
cnn_model.save("modelos/cnn_model_ok.keras")

In [ ]:
#  predicciones para el conjunto de validación
preds_cnn_val = cnn_model.predict(X_cnn_val).flatten()

47/47 ━━━━━━━━━━━━━━━━━━━━ 5s 102ms/step


In [ ]:
# Calculo del RMSE
rmse_cnn = np.sqrt(np.mean((y_cnn_val - preds_cnn_val)**2))
print("RMSE CNN:", rmse_cnn)

RMSE CNN: 2.1071088586605753
